# buckleUP — v2: retrofit + half-sine

Same chevron-braced bay and same corotational buckling physics as
`example_buckleUP.ipynb`, but two things change:

| | v1 | v2 |
|---|---|---|
| Build path | `add_imperfect_line` at construction time | `add_line` (clean) then `replace_line` retrofit |
| Shape | `'kink'` — one midspan interior point, 2 segments per brace | `'sine'` — half-sine envelope, 16 segments per brace |
| Accuracy | Triangular bent, all the L/1000 offset at midspan | `y(s) = δ₀ · sin(π·s/L)` matches the first Euler buckling mode exactly |

The **retrofit workflow** is the interesting bit. You build the frame
the way you normally would — every member is a plain `add_line`, no
imperfection math — and only *afterwards* do you decide which lines
should carry an imperfection. `replace_line` splices the imperfect
polyline in where the straight line used to be, preserves the
endpoints (so everything that referenced them still works), and
**automatically re-wires every physical group and label that contained
the old line** to contain the new segments. Nothing downstream has to
change.

The **half-sine shape** is more physically correct: the first
eigenmode of a pin-pin column under axial compression is a half-sine,
and seeding the imperfection with that same shape means the load is
amplifying *only* the mode that actually buckles. With a triangular
kink some of the imperfection energy sits in higher modes that never
activate, so the FEM curve is slightly stiffer than theory.

In [ ]:
from apeGmsh import apeGmsh, Results
import numpy as np
import matplotlib.pyplot as plt

# ---- Geometry [mm] ------------------------------------------------
L  = 4000.0     # bay width  (x)
H  = 3000.0     # storey height (y)
lc = 200.0      # mesh size

# ---- Steel material ----------------------------------------------
E   = 200_000.0      # MPa
nu  = 0.3

# ---- Column / beam section (weak in flexure → truss action) -----
A_frame = 10_000.0   # mm^2
I_frame = 1.0e4      # mm^4

# ---- Brace section — slender round bar d = 60 mm -----------------
d_br = 60.0
A_br = np.pi * (d_br / 2.0) ** 2
I_br = np.pi * d_br ** 4 / 64.0

# ---- Derived buckling numbers ------------------------------------
L_br       = np.sqrt((L / 2.0) ** 2 + H ** 2)
delta_0    = L_br / 1000.0
P_cr_brace = np.pi ** 2 * E * I_br / L_br ** 2
V_cr_ideal = P_cr_brace * L / L_br

print(f'L_brace    = {L_br:.1f} mm   delta_0 = L/1000 = {delta_0:.3f} mm')
print(f'A_brace    = {A_br:.1f} mm^2  I_brace = {I_br:.0f} mm^4')
print(f'P_cr per brace (pin-pin) = {P_cr_brace/1e3:.2f} kN')
print(f'V_cr_ideal               = {V_cr_ideal/1e3:.2f} kN')

## 1. Build the frame clean, then retrofit the braces

Step **(a)**: build every member as a plain straight line with
`add_line`. Labels, physical groups, everything exactly as you
would for a normal frame.

Step **(b)**: call `replace_line` on each brace to splice in a
`shape='sine'` imperfection with `n_segments=16` (the curve is
discretised into 16 line pieces following a half-sine envelope of
amplitude `L_br/1000`). The `direction=(±1, 0, 0)` hint is projected
perpendicular to each brace's axis automatically. After retrofit,
the old line tag is gone, the endpoint points are preserved, and
`pg_braces` + the `brace_{left,right}` labels now reference the new
polyline segments — no manual re-wiring.

In [ ]:
m = apeGmsh(model_name='buckleUP_v2', verbose=False)
m.begin()

# (a) --- Clean frame build with every member as a single add_line.
p_bL = m.model.geometry.add_point(0,   0, 0, lc=lc, label='base_left')
p_bR = m.model.geometry.add_point(L,   0, 0, lc=lc, label='base_right')
p_tL = m.model.geometry.add_point(0,   H, 0, lc=lc, label='top_left')
p_tR = m.model.geometry.add_point(L,   H, 0, lc=lc, label='top_right')
p_ap = m.model.geometry.add_point(L/2, H, 0, lc=lc, label='apex')

col_L  = m.model.geometry.add_line(p_bL, p_tL, label='col_left')
col_R  = m.model.geometry.add_line(p_bR, p_tR, label='col_right')
beam_L = m.model.geometry.add_line(p_tL, p_ap, label='beam_left')
beam_R = m.model.geometry.add_line(p_ap, p_tR, label='beam_right')
br_L   = m.model.geometry.add_line(p_bL, p_ap, label='brace_left')
br_R   = m.model.geometry.add_line(p_bR, p_ap, label='brace_right')

m.physical.add_curve(tags=[col_L, col_R],   name='pg_columns')
m.physical.add_curve(tags=[beam_L, beam_R], name='pg_beams')
m.physical.add_curve(tags=[br_L, br_R],     name='pg_braces')

print(f'clean build: br_L={br_L}, br_R={br_R}')
print(f'pg_braces before retrofit: '
      f'{[int(t) for t in m.labels.entities("brace_right", dim=1)]} (right), '
      f'{[int(t) for t in m.labels.entities("brace_left",  dim=1)]} (left)')

# (b) --- Retrofit: replace each straight brace with a half-sine polyline.
br_L_tags = m.model.geometry.replace_line(
    br_L,
    magnitude=delta_0,
    direction=(-1, 0, 0),     # outward on the left side
    shape='sine',
    n_segments=16,
)
br_R_tags = m.model.geometry.replace_line(
    br_R,
    magnitude=delta_0,
    direction=(+1, 0, 0),     # outward on the right side
    shape='sine',
    n_segments=16,
)

print(f'after retrofit: brace_left segs = {br_L_tags}')
print(f'                brace_right segs = {br_R_tags}')

# The brace labels and pg_braces now point to the new polyline segments.
print(f'pg_braces after retrofit: '
      f'{sorted(int(t) for t in m.labels.entities("brace_right", dim=1))} (right)')

m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=1)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'\ntotal nodes: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')

## 2. Verify the half-sine envelope

Walk every brace-right node from base to apex and compute its
perpendicular distance to the straight `(L, 0) → (L/2, H)` line.
For a half-sine imperfection the offset at arc-length fraction `s`
should be `δ₀ · sin(π·s)`, so we plot FEM points against the
analytical envelope.

In [ ]:
p_start = np.array([L,   0, 0])
p_end   = np.array([L/2, H, 0])
axis    = p_end - p_start
L_axis  = np.linalg.norm(axis)
u_axis  = axis / L_axis

br_R_ids    = np.array(
    [int(n) for n in fem.nodes.get_ids(target='brace_right')])
br_R_coords = fem.nodes.get_coords(target='brace_right')

s_frac  = np.zeros(len(br_R_ids))
offsets = np.zeros(len(br_R_ids))
for i, c in enumerate(br_R_coords):
    r = c - p_start
    s = np.dot(r, u_axis)
    s_frac[i]  = s / L_axis
    along = p_start + s * u_axis
    offsets[i] = np.linalg.norm(c - along)

order = np.argsort(s_frac)
s_frac  = s_frac[order]
offsets = offsets[order]

s_theory = np.linspace(0, 1, 200)
y_theory = delta_0 * np.sin(np.pi * s_theory)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(s_frac, offsets, 'o', color='tab:blue', label='FEM brace-right nodes')
ax.plot(s_theory, y_theory, 'k--', lw=1.0,
        label=f'δ₀·sin(π·s/L)  (δ₀ = {delta_0:.2f} mm)')
ax.set_xlabel('arc-length fraction s/L')
ax.set_ylabel('perpendicular offset [mm]')
ax.set_title('Right brace imperfection shape')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'max FEM perp offset = {offsets.max():.4f} mm  '
      f'(theory max = {delta_0:.4f} mm)')

## 3. OpenSees model (unchanged from v1)

Same setup as v1: linear transformation for columns/beams,
corotational for the braces, brace-end moment releases on elements
touching the base/apex joints so the brace behaves as a true pin-pin
member and the idealised `P_cr = π²EI/L²` applies.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 2, '-ndf', 3)
ops.timeSeries('Linear', 1)

for nid, xyz in fem.nodes.get():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]))

for label in ('base_left', 'base_right'):
    nid = int(fem.nodes.get_ids(target=label)[0])
    ops.fix(nid, 1, 1, 1)

ops.geomTransf('Linear',       1)
ops.geomTransf('Corotational', 2)

n_frame = 0
for group in fem.elements.get(target=['pg_columns', 'pg_beams']):
    for eid, conn in group:
        ni, nj = int(conn[0]), int(conn[1])
        ops.element('elasticBeamColumn', int(eid), ni, nj,
                    A_frame, E, I_frame, 1)
        n_frame += 1

joint_nids = {
    int(fem.nodes.get_ids(target='base_left')[0]),
    int(fem.nodes.get_ids(target='base_right')[0]),
    int(fem.nodes.get_ids(target='apex')[0]),
}

n_brace = 0
for group in fem.elements.get(target='pg_braces'):
    for eid, conn in group:
        ni, nj = int(conn[0]), int(conn[1])
        release = 0
        if ni in joint_nids:
            release = 1
        if nj in joint_nids:
            release = 2 if release == 0 else 3
        if release:
            ops.element('elasticBeamColumn', int(eid), ni, nj,
                        A_br, E, I_br, 2, '-release', release)
        else:
            ops.element('elasticBeamColumn', int(eid), ni, nj,
                        A_br, E, I_br, 2)
        n_brace += 1

print(f'frame elements (linear)             : {n_frame}')
print(f'brace elements (corotational, pin)  : {n_brace}')

## 4. Run the load-controlled analysis and capture a time-series

Identical solver settings to v1. Load factor ramped to 0.93·V_cr in
60 steps. Each step also captures the full displacement field so we
can feed it into `Results.from_fem` as a pseudo-time history for the
viewer.

In [ ]:
tR_nid = int(fem.nodes.get_ids(target='top_right')[0])

ops.pattern('Plain', 1, 1)
ops.load(tR_nid, 1.0, 0.0, 0.0)

ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-8, 200)
ops.algorithm('Newton')

V_target = 0.93 * V_cr_ideal
n_steps  = 60
dV       = V_target / n_steps
ops.integrator('LoadControl', dV)
ops.analysis('Static')

# Identify the brace-right midspan node (largest initial offset).
perp_offsets_R = np.array([
    np.linalg.norm(c - (p_start + np.dot(c - p_start, u_axis) * u_axis))
    for c in br_R_coords
])
mid_idx = int(np.argmax(perp_offsets_R))
mid_nid = int(br_R_ids[mid_idx])
print(f'brace-right midspan node = {mid_nid}  '
      f'initial perp offset = {perp_offsets_R[mid_idx]:.4f} mm')

perp_R = np.array([H, L/2, 0]) / L_axis

# Sample F_brace from one interior element of the right brace.
brace_R_element_ids = []
for group in fem.elements.get(target='brace_right'):
    for eid, _conn in group:
        brace_R_element_ids.append(int(eid))
probe_eid = brace_R_element_ids[len(brace_R_element_ids) // 2]

hist_V       = []
hist_uTR     = []
hist_mperp   = []
hist_Fbrace  = []
disp_per_step: list[np.ndarray] = []
n_nodes = len(fem.nodes.ids)

for step in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'failed at step {step + 1}')
        break
    V_now = (step + 1) * dV
    uTR   = ops.nodeDisp(tR_nid, 1)
    mdx   = ops.nodeDisp(mid_nid, 1)
    mdy   = ops.nodeDisp(mid_nid, 2)
    mperp = delta_0 + mdx * perp_R[0] + mdy * perp_R[1]
    F_br  = -ops.basicForce(probe_eid)[0]

    hist_V.append(V_now)
    hist_uTR.append(uTR)
    hist_mperp.append(mperp)
    hist_Fbrace.append(F_br)

    d_step = np.zeros((n_nodes, 3), dtype=np.float64)
    for i, nid in enumerate(fem.nodes.ids):
        di = ops.nodeDisp(int(nid))
        d_step[i, 0] = di[0]
        d_step[i, 1] = di[1]
    disp_per_step.append(d_step)

print(f'converged in {len(hist_V)} of {n_steps} steps')
print(f'final V            = {hist_V[-1]/1e3:.2f} kN  '
      f'({hist_V[-1]/V_cr_ideal*100:.1f}% V_cr_ideal)')
print(f'final F_brace      = {hist_Fbrace[-1]/1e3:.2f} kN  '
      f'({hist_Fbrace[-1]/P_cr_brace*100:.1f}% P_cr_brace)')
print(f'final midspan perp = {hist_mperp[-1]:.3f} mm  '
      f'(amplification = {hist_mperp[-1]/delta_0:.2f}x)')

## 5. Compare to Perry–Robertson

Same three-panel plot as v1. The FEM curve should hug the theory line
a little more closely than v1's triangular kink because the sine
imperfection seeds only the first buckling mode.

In [ ]:
hist_V      = np.asarray(hist_V)
hist_uTR    = np.asarray(hist_uTR)
hist_mperp  = np.asarray(hist_mperp)
hist_Fbrace = np.asarray(hist_Fbrace)

eta          = hist_Fbrace / P_cr_brace
valid        = eta < 0.995
delta_theory = delta_0 / (1.0 - eta[valid])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

ax = axes[0]
ax.plot(hist_uTR, hist_V / 1e3, 'o-', lw=1.4, ms=3, color='tab:blue')
ax.set_xlabel('top-right drift u_x [mm]')
ax.set_ylabel('V  [kN]')
ax.set_title('Lateral load vs. drift')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(hist_V / 1e3, hist_Fbrace / 1e3, 'o-', lw=1.4, ms=3,
        color='tab:green', label='F_brace from FEM')
ax.plot(hist_V / 1e3, hist_V * L_br / L / 1e3, 'k--', lw=1.0,
        label='V*L_br/L (pure truss)')
ax.axhline(P_cr_brace / 1e3, color='tab:red', lw=0.8, ls=':',
           label=f'P_cr = {P_cr_brace/1e3:.1f} kN')
ax.set_xlabel('V  [kN]')
ax.set_ylabel('F_brace  [kN]')
ax.set_title('Brace compression vs. frame load')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=9)

ax = axes[2]
ax.plot(hist_mperp, hist_Fbrace / 1e3, 'o-', lw=1.4, ms=3,
        color='tab:red', label='FEM (sine imperf, corotational)')
ax.plot(delta_theory, hist_Fbrace[valid] / 1e3, 'k--', lw=1.2,
        label='Perry-Robertson  \u03b4\u2080/(1-P/P_cr)')
ax.axhline(P_cr_brace / 1e3, color='k', lw=0.8, ls=':')
ax.axvline(delta_0, color='tab:gray', lw=0.8, ls=':',
           label=f'\u03b4\u2080 = L/1000 = {delta_0:.2f} mm')
ax.set_xlabel('right brace midspan perp deflection [mm]')
ax.set_ylabel('F_brace  [kN]')
ax.set_title('Half-sine amplification vs. theory')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Time-series viewer

Exactly as in v1: push every load step into `Results.from_fem` as a
pseudo-time frame and launch the apeGmsh viewer. Scrub through
`time = V` in newtons to watch the half-sine imperfection amplify.

In [ ]:
steps = []
for V_i, d_step in zip(hist_V, disp_per_step):
    u_mag = np.linalg.norm(d_step, axis=1)
    steps.append({
        'time': float(V_i),
        'point_data': {
            'Displacement': d_step,
            '|U|':          u_mag,
            'Ux':           d_step[:, 0],
            'Uy':           d_step[:, 1],
        },
    })

print(f'time-series steps: {len(steps)}')
print(f'time range       : {steps[0]["time"]:.1f} to {steps[-1]["time"]:.1f} N')

results = Results.from_fem(
    fem,
    steps=steps,
    include_pgs=['pg_columns', 'pg_beams', 'pg_braces'],
    name='buckleUP_v2',
)
results.viewer(blocking=False)